# Train CFGen on `cellarity_ibd_dataset_level_cleaned_1996`

This notebook can optionally train an encoder from scratch, then run CFGen with that checkpoint.

Before training, update the user settings cell:
- `DATASET_PATH`
- covariate fields (`COVARIATE_KEYS`, `THETA_COVARIATE`, `SIZE_FACTOR_COVARIATE`)
- `TRAIN_ENCODER_FROM_SCRATCH` and `ENCODER_CKPT`

In [31]:
from pathlib import Path
from pprint import pprint
import sys

import muon as mu
from omegaconf import OmegaConf


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "configs/configs_sccfm/train.yaml").exists() and (candidate / "cfgen").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing configs/configs_sccfm/train.yaml")


repo_root = find_repo_root(Path.cwd())
cfg_dir_cfgen = repo_root / "configs/configs_sccfm"
cfg_dir_encoder = repo_root / "configs/configs_encoder"

# Make the local package importable in notebooks even when not installed with pip -e .
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from cfgen.estimator.encoder_estimator import EncoderEstimator
from cfgen.estimator.cfgen_estimator import CfgenEstimator

print(f"Repo root: {repo_root}")

Repo root: /Users/skander/Documents/CFGen


In [32]:
# ---------------------------
# User settings (edit these)
# ---------------------------
import anndata as ad

DATASET_ID = "ibd_prompts2_raw"
adata = ad.read_h5ad(f"{DATASET_ID}.h5ad")
adata.obs


,sample,n_genes_by_counts,log1p_n_genes_by_counts,log1p_total_counts,tissue,cell_line,data_source_id,experiment_type,data_type,sample_id,...,vs2nd,vsAll,vs2nd_weighted,vsAll_weighted,scimilarity_predictions,tissue_unified,tissue_broad,disease_colon,disease_other,SCimilarity_level_4
SRX15110540_AAACCTGAGAAACGAG-1,SRX15110540,326,5.789960,6.401917,ileum,nan,GSE202052,in vivo,RNA-sc,GCA17,...,0.853659,0.70,0.851707,0.696415,"naive thymus-derived CD4-positive, alpha-beta ...",ileum,ileum,Crohn disease,none,lymphocyte
SRX15110540_AAACCTGAGCGAAGGG-1,SRX15110540,157,5.062595,6.352629,ileum,nan,GSE202052,in vivo,RNA-sc,GCA17,...,0.550000,0.44,0.559815,0.448274,memory B cell,ileum,ileum,Crohn disease,none,mature B cell
SRX15110540_AAACCTGGTGAAATCA-1,SRX15110540,130,4.875197,6.040255,ileum,nan,GSE202052,in vivo,RNA-sc,GCA17,...,0.959184,0.94,0.962139,0.942046,enterocyte,ileum,ileum,Crohn disease,none,enterocyte
SRX15110540_AAACCTGTCTCTGAGA-1,SRX15110540,183,5.214936,5.976351,ileum,nan,GSE202052,in vivo,RNA-sc,GCA17,...,0.882353,0.60,0.887056,0.610759,"CD4-positive, alpha-beta T cell",ileum,ileum,Crohn disease,none,lymphocyte
SRX15110540_AAACGGGAGAAACGAG-1,SRX15110540,308,5.733341,6.473891,ileum,nan,GSE202052,in vivo,RNA-sc,GCA17,...,0.938776,0.92,0.939253,0.920006,enterocyte,ileum,ileum,Crohn disease,none,enterocyte
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRX8130889_TTTGCGCTCTGCTTGC-1,SRX8130889,929,6.835185,7.682022,colon,NaN,GSE148837,in vivo,RNA-sc,S34_ulcerative_colitis,...,0.853659,0.70,0.849645,0.701419,"activated CD8-positive, alpha-beta T cell",colon,colon,ulcerative colitis,asthma,lymphocyte
SRX8130889_TTTGGTTCAAAGCAAT-1,SRX8130889,2746,7.918265,9.121072,colon,NaN,GSE148837,in vivo,RNA-sc,S34_ulcerative_colitis,...,0.880000,0.88,0.884016,0.884016,gamma-delta T cell,colon,colon,ulcerative colitis,asthma,lymphocyte
SRX8130889_TTTGGTTTCCAGATCA-1,SRX8130889,1595,7.375256,8.364275,colon,NaN,GSE148837,in vivo,RNA-sc,S34_ulcerative_colitis,...,0.789474,0.60,0.783545,0.608372,mucosal invariant T cell,colon,colon,ulcerative colitis,asthma,lymphocyte
SRX8130889_TTTGGTTTCTTTAGGG-1,SRX8130889,1816,7.504942,8.562166,colon,NaN,GSE148837,in vivo,RNA-sc,S34_ulcerative_colitis,...,0.650000,0.52,0.656798,0.524421,"double-positive, alpha-beta thymocyte",colon,colon,ulcerative colitis,asthma,lymphocyte


In [ ]:
NOTEBOOK_DIR = repo_root / "notebooks" / "train_cfgen"
DATASET_PATH = Path(f"{DATASET_ID}.h5ad")
if not DATASET_PATH.exists():
    DATASET_PATH = NOTEBOOK_DIR / f"{DATASET_ID}.h5ad"

# Dataset schema
USE_X_AS_INPUT = True
LAYER_KEY = "X" if USE_X_AS_INPUT else "X_counts"
COVARIATE_KEYS = ["tissue_unified", "disease_colon", "sample"]
THETA_COVARIATE = "tissue_unified"  # Must be one of COVARIATE_KEYS
SIZE_FACTOR_COVARIATE = "tissue_unified"  # Must be one of COVARIATE_KEYS
GUIDANCE_WEIGHTS = {"tissue_unified": 1.0, "disease_colon": 1.0, "sample": 1.0}
SUBSAMPLE_FRAC = 0.90  # Keep all cells so per-sample proportions are learned from the full data

# Encoder pretraining options
TRAIN_ENCODER_FROM_SCRATCH = True
ENCODER_CFG = "default"
ENCODER_PROJECT = f"encoder-{DATASET_ID}"
ENCODER_MAX_EPOCHS = 10

# CFGen training/runtime
DENOISING_CFG = "resnet_small"
USE_WANDB_OFFLINE = True
WANDB_PROJECT = f"cfgen-{DATASET_ID}"
ACCELERATOR = "auto"
DEVICES = 1
CFGEN_MAX_EPOCHS = 10
BATCH_SIZE = 5

# Used only when TRAIN_ENCODER_FROM_SCRATCH = False
ENCODER_CKPT = repo_root / "project_folder/experiments/autoencoder_ckpt/train_autoencoder_cellarity_ibd_dataset_level_cleaned_1996/checkpoints/last.ckpt"


In [34]:
print(f"Dataset file: {DATASET_PATH}")
assert DATASET_PATH.exists(), "Dataset file not found. Update DATASET_PATH."

adata_or_mudata = mu.read(str(DATASET_PATH))
if hasattr(adata_or_mudata, "mod"):
    obs = adata_or_mudata.mod["rna"].obs
    layers = list(adata_or_mudata.mod["rna"].layers.keys())
else:
    obs = adata_or_mudata.obs
    layers = list(adata_or_mudata.layers.keys())

print("First 30 obs columns:")
print(list(obs.columns[:30]))
print("\nAvailable layers:")
print(layers[:20])

if USE_X_AS_INPUT:
    print("\nUsing adata.X as input matrix.")
else:
    assert LAYER_KEY in layers, f"Layer '{LAYER_KEY}' not found in dataset."

for cov in COVARIATE_KEYS:
    assert cov in obs.columns, f"Missing covariate column: {cov}"
assert THETA_COVARIATE in obs.columns, f"Missing theta covariate: {THETA_COVARIATE}"
assert SIZE_FACTOR_COVARIATE in obs.columns, f"Missing size-factor covariate: {SIZE_FACTOR_COVARIATE}"

Dataset file: ibd_prompts2_raw.h5ad
First 30 obs columns:
['sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'tissue', 'cell_line', 'data_source_id', 'experiment_type', 'data_type', 'sample_id', 'donor_id', 'source_sample_id', 'sample_GSM', 'cell_enrichment_type', 'cell_enrichment', 'genotype', 'mouse_strain', 'model', 'disease', 'disease_status', 'treat', 'dose', 'dose_unit', 'time', 'time_unit', 'sex', 'bmi', 'total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes']

Available layers:
['norm']

Using adata.X as input matrix.


In [35]:
# Resolve encoder checkpoint: either train from scratch or use a provided checkpoint
if TRAIN_ENCODER_FROM_SCRATCH:
    print("Training encoder from scratch...")
    encoder_cfg = OmegaConf.create({
        "checkpoints": OmegaConf.load(cfg_dir_encoder / "checkpoints/default.yaml"),
        "early_stopping": OmegaConf.load(cfg_dir_encoder / "early_stopping/default.yaml"),
        "logger": OmegaConf.load(cfg_dir_encoder / "logger/default.yaml"),
        "trainer": OmegaConf.load(cfg_dir_encoder / "trainer/default.yaml"),
        "training_config": OmegaConf.load(cfg_dir_encoder / "training_config/default.yaml"),
        "encoder": OmegaConf.load(cfg_dir_encoder / f"encoder/{ENCODER_CFG}.yaml"),
    })

    encoder_cfg.dataset = OmegaConf.create({
        "dataset_path": str(DATASET_PATH),
        "layer_key": LAYER_KEY,
        "covariate_keys": COVARIATE_KEYS,
        "subsample_frac": SUBSAMPLE_FRAC,
        "normalization_type": "log_gexp",
        "is_binarized": False,
        "theta_covariate": THETA_COVARIATE,
        "split_rates": [0.9, 0.1],
    })

    encoder_cfg.training_config.batch_size = BATCH_SIZE
    encoder_cfg.training_config.use_early_stopping = False
    encoder_cfg.logger.offline = USE_WANDB_OFFLINE
    encoder_cfg.logger.project = ENCODER_PROJECT
    encoder_cfg.trainer.accelerator = ACCELERATOR
    encoder_cfg.trainer.devices = DEVICES
    encoder_cfg.trainer.max_epochs = ENCODER_MAX_EPOCHS

    encoder_estimator = EncoderEstimator(encoder_cfg)
    encoder_estimator.train()

    resolved_encoder_ckpt = encoder_estimator.training_dir / "checkpoints" / "last.ckpt"
    assert resolved_encoder_ckpt.exists(), "Encoder training finished but last.ckpt was not found."
else:
    resolved_encoder_ckpt = Path(ENCODER_CKPT)
    assert resolved_encoder_ckpt.exists(), "Encoder checkpoint not found. Update ENCODER_CKPT."

print("Encoder checkpoint:", resolved_encoder_ckpt)

# Build CFGen base config blocks from repo defaults
cfg = OmegaConf.create({
    "checkpoints": OmegaConf.load(cfg_dir_cfgen / "checkpoints/default.yaml"),
    "denoising_module": OmegaConf.load(cfg_dir_cfgen / f"denoising_module/{DENOISING_CFG}.yaml"),
    "early_stopping": OmegaConf.load(cfg_dir_cfgen / "early_stopping/default.yaml"),
    "generative_model": OmegaConf.load(cfg_dir_cfgen / "generative_model/default.yaml"),
    "logger": OmegaConf.load(cfg_dir_cfgen / "logger/default.yaml"),
    "trainer": OmegaConf.load(cfg_dir_cfgen / "trainer/default.yaml"),
    "training_config": OmegaConf.load(cfg_dir_cfgen / "training_config/default.yaml"),
    "encoder": OmegaConf.load(cfg_dir_cfgen / f"encoder/{ENCODER_CFG}.yaml"),
})

# Inject dataset config directly (no new Hydra dataset YAML file required)
cfg.dataset = OmegaConf.create({
    "dataset_path": str(DATASET_PATH),
    "layer_key": LAYER_KEY,
    "covariate_keys": COVARIATE_KEYS,
    "subsample_frac": SUBSAMPLE_FRAC,
    "normalization_type": "log_gexp",
    "one_hot_encode_features": False,
    "split_rates": [0.9, 0.1],
    "cov_embedding_dimensions": cfg.denoising_module.embedding_dim,
    "is_binarized": False,
    "theta_covariate": THETA_COVARIATE,
    "size_factor_covariate": SIZE_FACTOR_COVARIATE,
    "guidance_weights": GUIDANCE_WEIGHTS,
})

cfg.training_config.encoder_ckpt = str(resolved_encoder_ckpt)
cfg.training_config.batch_size = BATCH_SIZE
cfg.training_config.use_early_stopping = False
cfg.logger.offline = USE_WANDB_OFFLINE
cfg.logger.project = WANDB_PROJECT
cfg.trainer.accelerator = ACCELERATOR
cfg.trainer.devices = DEVICES
cfg.trainer.max_epochs = CFGEN_MAX_EPOCHS

print(f"\nUsing subsample_frac={SUBSAMPLE_FRAC}")
print("\nFinal CFGen config:")
print(OmegaConf.to_yaml(cfg))

Training encoder from scratch...
Create the training folders...
Initialize data module...


You have turned on `Trainer(detect_anomaly=True)`. This will significantly slow down compute speed and is recommended only for model debugging.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.


Initialize model...
Encoder architecture EncoderModel(
  (encoder): ModuleDict(
    (rna): MLP(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=1996, out_features=512, bias=True)
          (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ELU(alpha=1.0)
        )
        (1): Sequential(
          (0): Linear(in_features=512, out_features=256, bias=True)
          (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ELU(alpha=1.0)
        )
        (2): Linear(in_features=256, out_features=50, bias=True)
      )
    )
  )
  (decoder): ModuleDict(
    (rna): MLP(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=50, out_features=256, bias=True)
          (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ELU(alpha=1.0)
        )
        (1): Sequential(
          (0): Lin

┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder      │ ModuleDict │  1.2 M │ train │     0 │
│ 1 │ decoder      │ ModuleDict │  1.2 M │ train │     0 │
│   │ other params │ n/a        │  2.0 K │ n/a   │   n/a │
└───┴──────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 2.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.3 M                                                                                                
Total estimated model params size (MB): 9                                                                          
Modules in train mode: 24                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


`Trainer.fit` stopped: `max_epochs=2` reached.


Encoder checkpoint: /Users/skander/Documents/CFGen/project_folder/experiments/encoder-ibd_prompts2_raw/1e390eb1-881a-45c8-b832-485d44f44af5/checkpoints/last.ckpt

Using subsample_frac=0.1

Final CFGen config:
checkpoints:
  filename: epoch_{epoch:01d}
  monitor: valid/loss
  mode: min
  every_n_epochs: 50
  save_last: true
  auto_insert_metric_name: false
denoising_module:
  denoising_net: resnet
  hidden_dim: 32
  dropout_prob: 0
  n_blocks: 3
  embedding_dim: 20
  conditional: true
  normalization: none
  is_binarized: false
  embed_size_factor: false
  conditioning_probability: 0.8
  guided_conditioning: true
early_stopping:
  monitor: valid/loss
  patience: 10000
  mode: min
  min_delta: 0.0
  verbose: false
  strict: true
  check_finite: true
  stopping_threshold: null
  divergence_threshold: null
  check_on_train_epoch_end: null
generative_model:
  learning_rate: 0.0001
  weight_decay: 1.0e-06
  antithetic_time_sampling: true
  sigma: 0.0001
  use_ot: false
logger:
  offline: tru

In [36]:
# Initialize model and data
estimator = CfgenEstimator(cfg)

total_params = sum(p.numel() for p in estimator.generative_model.parameters())
trainable_params = sum(p.numel() for p in estimator.generative_model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Create the training folders...
Initialize data module...


You have turned on `Trainer(detect_anomaly=True)`. This will significantly slow down compute speed and is recommended only for model debugging.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:213: Attribute 'encoder_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['encoder_model'])`.
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:213: Attribute 'denoising_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them 

Initialize feature embeddings...
Initialize model...
Denoising model MLPTimeStep(
  (time_embedder): Sequential(
    (0): Linear(in_features=20, out_features=20, bias=True)
    (1): SiLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
  )
  (net_in): Linear(in_features=50, out_features=32, bias=True)
  (blocks): ModuleList(
    (0-2): 3 x ResnetBlock(
      (net1): Sequential(
        (0): SiLU()
        (1): Linear(in_features=32, out_features=32, bias=True)
      )
      (cond_proj): Sequential(
        (0): SiLU()
        (1): Linear(in_features=20, out_features=32, bias=True)
      )
      (net2): Sequential(
        (0): SiLU()
        (1): Linear(in_features=32, out_features=32, bias=True)
      )
    )
  )
  (net_out): Sequential(
    (0): SiLU()
    (1): Linear(in_features=32, out_features=50, bias=True)
  )
)
Encoder architecture EncoderModel(
  (encoder): ModuleDict(
    (rna): MLP(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_featu

In [37]:
# Train CFGen
estimator.train()

/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_model   │ EncoderModel │  2.3 M │ eval  │     0 │
│ 1 │ denoising_model │ MLPTimeStep  │ 12.5 K │ train │     0 │
│ 2 │ criterion       │ MSELoss      │      0 │ train │     0 │
└───┴─────────────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 12.5 K                                                                                           
Non-trainable params: 2.3 M                                                                                        
Total params: 2.4 M                                                                                                
Total estimated model params size (MB): 9                                                                          
Modules in train mode: 41                                                                                          
Modules in eval mode: 25                                                                                           
Total FLOPs: 0

Output()

/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

...

/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


`Trainer.fit` stopped: `max_epochs=2` reached.


In [38]:
# Evaluate
import importlib
import cfgen.models.fm.fm as fm_module

# Reload latest FM implementation from disk and hot-patch the in-memory model class
importlib.reload(fm_module)
type(estimator.generative_model).sample = fm_module.FM.sample

estimator.test()
metrics = {k: (float(v.detach().cpu()) if hasattr(v, "detach") else v) for k, v in estimator.trainer_generative.callback_metrics.items()}
pprint(metrics)

/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:429: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.


Output()

/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/scipy/sparse/_index.py:210: 
SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more 
efficient.
  self._set_arrayXarray(i, j, x)

/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: 
ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)

/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/scipy/sparse/_index.py:210: 
SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more 
efficient.
  self._set_arrayXarray(i, j, x)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│     1-Wasserstein_rna     │     80.38528442382812     │
│     2-Wasserstein_rna     │     583.5529174804688     │
│      Linear_MMD_rna       │     119.7857894897461     │
│        RBF_MMD_rna        │    1.1563191413879395     │
└───────────────────────────┴───────────────────────────┘

{'1-Wasserstein_rna': 80.38528442382812,
 '2-Wasserstein_rna': 583.5529174804688,
 'Linear_MMD_rna': 119.7857894897461,
 'RBF_MMD_rna': 1.1563191413879395}


In [39]:
# Generate from explicit prompts, conditioning on tissue/disease/sample
import torch
import pandas as pd
import anndata as ad
import muon as mu
import numpy as np

N_PER_PROMPT = 2000
N_SAMPLE_STEPS = 10
PROMPTS = [
    {"tissue_unified": "ileum", "disease_colon": "Crohn disease"},
    {"tissue_unified": "colon", "disease_colon": "ulcerative colitis"},
]

gm = estimator.generative_model
sample_device = next(gm.parameters()).device

gm.denoising_model = gm.denoising_model.to(sample_device)
gm.encoder_model = gm.encoder_model.to(sample_device)
for cov_name in gm.feature_embeddings:
    gm.feature_embeddings[cov_name] = gm.feature_embeddings[cov_name].to(sample_device)

def move_nested_tensors_to_device(obj, device):
    if torch.is_tensor(obj):
        return obj.to(device)
    if isinstance(obj, dict):
        return {k: move_nested_tensors_to_device(v, device) for k, v in obj.items()}
    return obj

gm.size_factor_statistics = move_nested_tensors_to_device(gm.size_factor_statistics, sample_device)
print("Sampling device:", sample_device)

id2cov = estimator.dataset.id2cov
required_prompt_keys = ["tissue_unified", "disease_colon", "sample"]
for key in required_prompt_keys:
    if key not in id2cov and not (key == "tissue_unified" and "tissue" in id2cov):
        raise KeyError(f"Prompt key '{key}' not available in model covariates: {list(id2cov.keys())}")

# Gene names in the exact order the model was trained on.
reference_var_names = estimator.dataset.var_names["rna"]
n_reference_genes = len(reference_var_names)
print(f"Reference gene count (from training dataset): {n_reference_genes}")
assert n_reference_genes == estimator.dataset.X["rna"].shape[1],     "var_names length does not match the gene dimension the model was trained on!"

adata_obj = mu.read(str(DATASET_PATH))
if hasattr(adata_obj, "mod") and "rna" in adata_obj.mod:
    obs_map = adata_obj.mod["rna"].obs
else:
    obs_map = adata_obj.obs

required_obs_cols = ["tissue_unified", "disease_colon", "sample"]
missing_obs_cols = [c for c in required_obs_cols if c not in obs_map.columns]
if missing_obs_cols:
    raise KeyError(f"Missing required columns in adata.obs: {missing_obs_cols}")

if "tissue_unified" in id2cov:
    tissue_cov_key = "tissue_unified"
else:
    if "tissue" not in id2cov:
        raise KeyError(
            "Model covariates contain neither 'tissue_unified' nor 'tissue'. "
            f"Available: {list(id2cov.keys())}"
        )
    if "tissue" not in obs_map.columns:
        raise KeyError("Cannot map tissue_unified to model key because 'tissue' is missing in adata.obs.")
    tissue_cov_key = "tissue"

conditioning_covariates = [tissue_cov_key, "disease_colon", "sample"]
guidance_weights_generation = {
    tissue_cov_key: GUIDANCE_WEIGHTS.get("tissue_unified", 1.0),
    "disease_colon": GUIDANCE_WEIGHTS.get("disease_colon", 1.0),
    "sample": GUIDANCE_WEIGHTS.get("sample", 1.0),
}

print(f"Using model tissue key: {tissue_cov_key}")

def get_covariate_id(cov_name: str, cov_value: str) -> int:
    mapping = id2cov[cov_name]
    if cov_value in mapping:
        return int(mapping[cov_value])
    mapping_str = {str(k): int(v) for k, v in mapping.items()}
    if str(cov_value) in mapping_str:
        return mapping_str[str(cov_value)]
    raise KeyError(f"Value '{cov_value}' not found for '{cov_name}'. Available: {sorted(mapping_str.keys())}")

def map_tissue_value_to_model_value(tissue_unified_value: str) -> str:
    if tissue_cov_key == "tissue_unified":
        return tissue_unified_value

    tissue_values = sorted(
        obs_map.loc[obs_map["tissue_unified"].astype(str) == str(tissue_unified_value), "tissue"].astype(str).unique()
    )
    if len(tissue_values) == 0:
        raise KeyError(f"No rows found for tissue_unified='{tissue_unified_value}' in adata.obs.")
    if len(tissue_values) > 1:
        raise ValueError(f"Ambiguous mapping tissue_unified -> tissue for '{tissue_unified_value}': {tissue_values}")
    return tissue_values[0]

def allocate_counts_by_proportion(counts: pd.Series, total: int) -> pd.Series:
    proportions = counts / counts.sum()
    raw = proportions * total
    base = np.floor(raw).astype(int)
    remainder = int(total - base.sum())
    if remainder > 0:
        frac = raw - base
        top_idx = frac.sort_values(ascending=False).index[:remainder]
        base.loc[top_idx] += 1
    return base

all_adatas = []

gm.eval()
with torch.no_grad():
    for i, prompt in enumerate(PROMPTS, start=1):
        tissue_unified_value = str(prompt["tissue_unified"])
        disease_value = str(prompt["disease_colon"])

        tissue_model_value = map_tissue_value_to_model_value(tissue_unified_value)
        tissue_id = get_covariate_id(tissue_cov_key, tissue_model_value)
        disease_id = get_covariate_id("disease_colon", disease_value)

        pair_mask = (
            obs_map["tissue_unified"].astype(str).eq(tissue_unified_value)
            & obs_map["disease_colon"].astype(str).eq(disease_value)
        )
        pair_obs = obs_map.loc[pair_mask].copy()
        if pair_obs.empty:
            raise ValueError(
                f"No real cells found for tissue_unified='{tissue_unified_value}', disease_colon='{disease_value}'."
            )

        pair_obs["sample"] = pair_obs["sample"].astype(str)
        sample_counts_real = pair_obs["sample"].value_counts().sort_index()
        sample_counts_gen = allocate_counts_by_proportion(sample_counts_real, N_PER_PROMPT)
        sample_counts_gen = sample_counts_gen[sample_counts_gen > 0]

        prompt_adatas = []
        for sample_value, n_sample in sample_counts_gen.items():
            sample_id = get_covariate_id("sample", str(sample_value))

            covariate_indices = {
                tissue_cov_key: torch.full((int(n_sample),), tissue_id, dtype=torch.long, device=sample_device),
                "disease_colon": torch.full((int(n_sample),), disease_id, dtype=torch.long, device=sample_device),
                "sample": torch.full((int(n_sample),), sample_id, dtype=torch.long, device=sample_device),
            }

            generated = gm.sample(
                batch_size=int(n_sample),
                n_sample_steps=N_SAMPLE_STEPS,
                theta_covariate=THETA_COVARIATE,
                size_factor_covariate=SIZE_FACTOR_COVARIATE,
                conditioning_covariates=conditioning_covariates,
                covariate_indices=covariate_indices,
                unconditional=False,
                guidance_weights=guidance_weights_generation,
            )

            X_generated = generated["rna"].detach().cpu().numpy()
            if X_generated.shape[1] != n_reference_genes:
                raise ValueError(
                    f"Generated matrix has {X_generated.shape[1]} genes but the model "
                    f"was trained on {n_reference_genes} genes. Dimension mismatch."
                )

            obs_generated = pd.DataFrame({
                "tissue_unified": [tissue_unified_value] * int(n_sample),
                "disease_colon": [disease_value] * int(n_sample),
                "sample": [str(sample_value)] * int(n_sample),
                tissue_cov_key: [tissue_model_value] * int(n_sample),
            })

            adata_sample = ad.AnnData(
                X=X_generated,
                obs=obs_generated,
                var=pd.DataFrame(index=reference_var_names),
            )
            prompt_adatas.append(adata_sample)

        adata_prompt = ad.concat(prompt_adatas, axis=0, merge="same")
        adata_prompt = adata_prompt[:, reference_var_names].copy()
        all_adatas.append(adata_prompt)

        tissue_slug = tissue_unified_value.replace(" ", "_")
        disease_slug = disease_value.replace(" ", "_")
        out_path_cond = NOTEBOOK_DIR / f"generated_prompt_{i}_{tissue_slug}_{disease_slug}.h5ad"
        adata_prompt.write_h5ad(out_path_cond)

        sample_split_str = ", ".join([f"{s}:{int(n)}" for s, n in sample_counts_gen.items()])
        print(
            f"Generated {adata_prompt.n_obs} cells for prompt {i}: "
            f"tissue_unified='{tissue_unified_value}', disease_colon='{disease_value}' "
            f"with sample split [{sample_split_str}] -> {out_path_cond}"
        )

if not all_adatas:
    raise RuntimeError("No prompt outputs were generated.")

adata_generated = ad.concat(all_adatas, axis=0, merge="same")
adata_generated = adata_generated[:, reference_var_names].copy()
print("Combined generated matrix shape:", adata_generated.shape)
print("Generated obs columns:", list(adata_generated.obs.columns))

out_path = NOTEBOOK_DIR / "generated_prompts2_ileum_crohn_colon_uc.h5ad"
adata_generated.write_h5ad(out_path)
print("Saved combined file:", out_path)



Sampling device: cpu
Reference gene count (from training dataset): 1996


Using model tissue key: tissue_unified


/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/alig

Generated 2000 cells for prompt 1: tissue_unified='ileum', disease_colon='Crohn disease' with sample split [SRX15110540:89, SRX15110541:142, SRX15110542:183, SRX15110543:131, SRX15110544:287, SRX15110545:221, SRX15110546:189, SRX15110547:142, SRX15110549:55, SRX15110550:133, SRX15110551:57, SRX15110562:81, SRX15110563:128, SRX15110566:162] -> /Users/skander/Documents/CFGen/notebooks/train_cfgen/generated_prompt_1_ileum_Crohn_disease.h5ad


/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/alig

Generated 2000 cells for prompt 2: tissue_unified='colon', disease_colon='ulcerative colitis' with sample split [SRX11801297:87, SRX11801298:179, SRX11801304:96, SRX18901467:239, SRX18901468:195, SRX18901469:194, SRX18901470:180, SRX4079052:108, SRX4079053:133, SRX4291258:39, SRX4291259:45, SRX4291261:32, SRX4291262:65, SRX4291264:79, SRX4291265:85, SRX6486669:45, SRX6486670:35, SRX6486671:42, SRX8130887:45, SRX8130888:35, SRX8130889:42] -> /Users/skander/Documents/CFGen/notebooks/train_cfgen/generated_prompt_2_colon_ulcerative_colitis.h5ad
Combined generated matrix shape: (4000, 1996)
Generated obs columns: ['tissue_unified', 'disease_colon', 'sample']
Saved combined file: /Users/skander/Documents/CFGen/notebooks/train_cfgen/generated_prompts2_ileum_crohn_colon_uc.h5ad


/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/Users/skander/miniconda3/envs/cfgen/lib/python3.10/site-packages/anndat

In [40]:
adata_generated.obs

,tissue_unified,disease_colon,sample
0,ileum,Crohn disease,SRX15110540
1,ileum,Crohn disease,SRX15110540
2,ileum,Crohn disease,SRX15110540
3,ileum,Crohn disease,SRX15110540
4,ileum,Crohn disease,SRX15110540
...,...,...,...
37,colon,ulcerative colitis,SRX8130889
38,colon,ulcerative colitis,SRX8130889
39,colon,ulcerative colitis,SRX8130889
40,colon,ulcerative colitis,SRX8130889
